# Preparación del entorno de entrenamiento con GPU

Proyecto: Clasificación de defectos superficiales en palta Hass (PICADURA y RUSSET).

Este registro describe la preparación de Ubuntu WSL2, Python, TensorFlow y CUDA para utilizar la tarjeta NVIDIA en los siguientes entrenamientos de ResNet50. Los comandos y resultados de las secciones iniciales corresponden a pruebas realizadas en la terminal. Las celdas de diagnóstico del final son opcionales.

No se modificaron los archivos originales del dataset durante esta configuración.

## 1. Estado alcanzado

| Elemento | Resultado comprobado |
|---|---|
| Sistema principal | Windows, con Visual Studio Code |
| Linux | Ubuntu 26.04 LTS sobre WSL2 |
| GPU | NVIDIA GeForce RTX 3070 Ti Laptop GPU |
| Entorno de Python en Windows | `C:\Users\TCI_AUTO\venvs\palta` (Python 3.11.9) |
| Entorno de Python en Ubuntu | `/home/tci_auto/.venvs/palta_gpu` (Python 3.12.14) |
| TensorFlow de Ubuntu | 2.21.0 |
| CUDA y cuDNN | Paquetes instalados junto con TensorFlow; bibliotecas localizadas mediante `LD_LIBRARY_PATH` |
| TensorFlow en Ubuntu | Detectó `/physical_device:GPU:0` con las rutas CUDA habilitadas |
| Prueba de cálculo | Matriz de 1024 × 1024 multiplicada en GPU; resultado 1024.0 |
| Entrenamiento nuevo | Aún no se ejecutó D1, D2 ni fine-tuning en este entorno |

La detección y el cálculo en GPU se verificaron desde la terminal de Ubuntu. Todavía falta configurar el acceso desde el kernel Jupyter de VS Code y dejar resuelta la carga de bibliotecas en futuras sesiones.

## 2. ¿Qué es CUDA?

CUDA es una plataforma de cómputo desarrollada por NVIDIA que permite aprovechar la GPU en operaciones matemáticas de propósito general. TensorFlow utiliza bibliotecas CUDA para realizar cálculos sobre la tarjeta gráfica. cuDNN proporciona implementaciones optimizadas de operaciones habituales en redes neuronales, como convoluciones.

En este proyecto, ResNet50 se implementa con TensorFlow/Keras. La GPU puede acelerar el procesamiento de lotes y el ajuste de pesos, especialmente cuando se habilite el fine-tuning. La mejora de tiempo dependerá del tamaño de las imágenes, el lote, la lectura de los TIFF y la memoria disponible.

## 3. Arquitectura del entorno

```text
Equipo con Windows
|
|-- VS Code
|   |-- PROYECTO_TI_2
|   |   |-- 1. fuentes/
|   |   |-- 6. entrenamiento/resnet50.ipynb
|   |   `-- 7. resultados/
|   `-- Entorno Python palta (Windows, CPU)
|
|-- DATASET_PALTA  (ubicado fuera del repositorio)
|
`-- WSL2
    `-- Ubuntu 26.04
        |-- Python 3.12.14
        |-- Entorno ~/.venvs/palta_gpu
        |-- TensorFlow 2.21.0 / Keras
        |-- Bibliotecas CUDA y cuDNN
        |-- Acceso al controlador NVIDIA de WSL2
        `-- NVIDIA RTX 3070 Ti Laptop GPU
```

Flujo de cálculo:

```text
Notebook -> Python -> TensorFlow/Keras -> CUDA/cuDNN -> GPU NVIDIA
```

El editor puede seguir funcionando en Windows, pero para ejecutar con GPU el notebook debe utilizar un proceso Python de Ubuntu con acceso a las bibliotecas CUDA. Seleccionar el kernel `palta` de Windows no activa la GPU.

## 4. Ubicación del proyecto y de los datos

Ruta del proyecto en Windows:

```text
C:\Users\TCI_AUTO\Documents\Maestría UNI\4_2. Proyecto de Invetigación  2\PROYECTO_TI_2
```

Ruta del dataset en Windows:

```text
C:\Users\TCI_AUTO\Documents\Maestría UNI\4_2. Proyecto de Invetigación  2\DATASET_PALTA
```

Ubuntu accede a las mismas carpetas a través de `/mnt/c/`:

```text
/mnt/c/Users/TCI_AUTO/Documents/Maestría UNI/4_2. Proyecto de Invetigación  2/PROYECTO_TI_2
/mnt/c/Users/TCI_AUTO/Documents/Maestría UNI/4_2. Proyecto de Invetigación  2/DATASET_PALTA
```

Los datos no se copiaron a Linux ni se añadieron a GitHub. Para los entrenamientos se deben conservar juntos los registros de las treinta vistas de cada fruto y mantener la separación entre train, validation y test.

## 5. Comprobaciones realizadas

### 5.1. Confirmación de WSL2

En PowerShell:

```powershell
wsl --list --verbose
```

Resultado observado:

```text
NAME      STATE     VERSION
Ubuntu    Stopped   2
```

El estado `Stopped` solo indica que Ubuntu no estaba ejecutándose en ese momento. Se inició al invocar WSL.

### 5.2. Acceso a la GPU desde Ubuntu

En PowerShell:

```powershell
wsl -d Ubuntu -- nvidia-smi
```

`nvidia-smi` respondió dentro de Ubuntu y mostró información de NVIDIA. Inicialmente no había procesos utilizando la GPU.

### 5.3. Python disponible en Ubuntu

Comando utilizado desde PowerShell:

```powershell
wsl -d Ubuntu -- bash -lc "python3 --version; command -v python3; python3 -m pip --version"
```

Salida observada:

```text
Python 3.14.4
/usr/bin/python3
/usr/bin/python3: No module named pip
```

El Python del sistema no se modificó. Después se comprobó que estaban disponibles Python 3.12 y `uv`:

```text
PRETTY_NAME="Ubuntu 26.04 LTS"
/home/tci_auto/.local/bin/python3.12
/home/tci_auto/.local/bin/uv
```

La ausencia de `pip` en el Python del sistema no impide trabajar con un entorno virtual creado mediante `uv`.

### 5.4. Creación del entorno de entrenamiento

Desde PowerShell:

```powershell
wsl -d Ubuntu -- bash -lc "mkdir -p ~/.venvs && uv venv --python 3.12 ~/.venvs/palta_gpu && ~/.venvs/palta_gpu/bin/python --version"
```

Resultado:

```text
Using CPython 3.12.14
Creating virtual environment at: /home/tci_auto/.venvs/palta_gpu
Python 3.12.14
```

Este entorno es independiente del Python de Windows y está ubicado dentro del sistema de archivos de Ubuntu.

### 5.5. Instalación de TensorFlow y dependencias de GPU

Desde PowerShell:

```powershell
wsl -d Ubuntu -- bash -lc "uv pip install --python ~/.venvs/palta_gpu/bin/python 'tensorflow[and-cuda]==2.21.0'"
```

La instalación terminó con 44 paquetes. Entre los principales se registraron TensorFlow 2.21.0, Keras 3.15.1, CUDA Runtime 12.9 y cuDNN 9.27. La instalación de los paquetes no fue suficiente, por sí sola, para que TensorFlow encontrara todas las bibliotecas compartidas.

### 5.6. Primer diagnóstico de TensorFlow

La consulta inicial devolvió TensorFlow 2.21.0, pero únicamente detectó la CPU:

```text
Cannot dlopen some GPU libraries.
Skipping registering GPU devices...
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
```

El controlador de NVIDIA sí era accesible mediante WSL2. El siguiente diagnóstico consistió en añadir temporalmente al cargador de Linux las rutas de las bibliotecas CUDA instaladas dentro del entorno virtual.

### 5.7. Habilitación temporal de las rutas de bibliotecas

Para evitar errores de comillas desde PowerShell, primero se ingresó a Ubuntu:

```powershell
wsl -d Ubuntu
```

Los dos comandos siguientes se ejecutaron directamente en la terminal de Ubuntu, no en PowerShell:

```bash
LIBS=$(find "$HOME/.venvs/palta_gpu/lib/python3.12/site-packages/nvidia" -type d -name lib -print | paste -sd: -)
```

```bash
LD_LIBRARY_PATH="$LIBS:/usr/lib/wsl/lib:${LD_LIBRARY_PATH:-}" "$HOME/.venvs/palta_gpu/bin/python" -c 'import tensorflow as tf; print(tf.config.list_physical_devices("GPU"))'
```

Resultado observado:

```text
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
```

`LD_LIBRARY_PATH` se aplicó solo al proceso lanzado con ese comando. No se ha configurado todavía de manera permanente para todas las terminales o kernels Jupyter.

### 5.8. Validación de operaciones reales en GPU

Desde Ubuntu y utilizando el valor de `LIBS` definido en la sesión:

```bash
LD_LIBRARY_PATH="$LIBS:/usr/lib/wsl/lib:${LD_LIBRARY_PATH:-}" "$HOME/.venvs/palta_gpu/bin/python" -c 'import tensorflow as tf; a=tf.ones((1024,1024)); b=tf.matmul(a,a); print("Dispositivo utilizado:",b.device); print("Resultado:",b[0,0].numpy())'
```

Salida relevante:

```text
Created device ... /device:GPU:0 with 5561 MB memory
name: NVIDIA GeForce RTX 3070 Ti Laptop GPU
Dispositivo utilizado: /job:localhost/replica:0/task:0/device:GPU:0
Resultado: 1024.0
```

Con esta prueba se comprobó que la operación se ejecutó en la GPU y produjo el valor esperado. Los 5561 MB corresponden a memoria anunciada como disponible para ese proceso, no necesariamente a toda la capacidad física de la tarjeta.

## 6. Alcance de la validación

Se verificó el acceso de Ubuntu a NVIDIA y la ejecución de operaciones de TensorFlow sobre GPU. No se ha ejecutado todavía el entrenamiento D1 o D2 ni fine-tuning dentro de WSL2.

Tampoco se ha validado aún que el kernel de Jupyter iniciado desde VS Code herede correctamente las rutas CUDA. Ese es el próximo trabajo antes de entrenar.

Las salidas con `oneDNN` y los mensajes sobre instrucciones de CPU fueron informativos. El diagnóstico relevante fue primero el error de carga de bibliotecas y después la confirmación de `GPU:0`.

## 7. Verificación opcional desde este notebook

Estas celdas son diagnósticas. No descargan paquetes, no modifican el dataset y no entrenan modelos.

Si el notebook se abre con el kernel de Windows, puede informar que TensorFlow no encuentra ninguna GPU. Para probar la GPU de WSL2 se deberá abrir el proyecto desde Ubuntu y seleccionar el entorno `palta_gpu`, con las rutas CUDA configuradas antes de iniciar Jupyter.

In [1]:
import os
import platform
import sys
from pathlib import Path

print("Sistema:", platform.platform())
print("Intérprete:", sys.executable)
print("WSL:", os.environ.get("WSL_DISTRO_NAME", "No indicado"))

base = (Path("/mnt/c/Users/TCI_AUTO/Documents/Maestría UNI/4_2. Proyecto de Invetigación  2")
        if platform.system() == "Linux" else
        Path(r"C:\Users\TCI_AUTO\Documents\Maestría UNI\4_2. Proyecto de Invetigación  2"))

print("Proyecto encontrado:", (base / "PROYECTO_TI_2").is_dir())
print("Dataset encontrado:", (base / "DATASET_PALTA").is_dir())

Sistema: Windows-10-10.0.26200-SP0
Intérprete: c:\Users\TCI_AUTO\venvs\palta\Scripts\python.exe
WSL: No indicado
Proyecto encontrado: True
Dataset encontrado: True


In [2]:
try:
    import tensorflow as tf
except ImportError as exc:
    print("TensorFlow no está disponible en este kernel:", exc)
else:
    print("TensorFlow:", tf.__version__)
    gpus = tf.config.list_physical_devices("GPU")
    print("GPU detectadas:", gpus)
    if gpus:
        with tf.device("/GPU:0"):
            matriz = tf.ones((1024, 1024), dtype=tf.float32)
            producto = tf.matmul(matriz, matriz)
        print("Dispositivo del cálculo:", producto.device)
        print("Resultado de comprobación:", producto[0, 0].numpy())
    else:
        print("El kernel actual no dispone de GPU; no se ejecutó la prueba de matriz.")

TensorFlow: 2.21.0
GPU detectadas: []
El kernel actual no dispone de GPU; no se ejecutó la prueba de matriz.


## 8. Próximas tareas

1. Revisar las dependencias necesarias para el notebook `resnet50.ipynb` en `palta_gpu`, incluyendo lectura de TIFF, métricas y gráficos. TensorFlow está instalado; el resto no se ha comprobado por completo.
2. Preparar el kernel Jupyter en Ubuntu y asegurar que el proceso herede las rutas CUDA. No confundir el kernel `palta` de Windows con el entorno `palta_gpu` de Ubuntu.
3. Ejecutar las celdas de verificación y confirmar un cálculo en `GPU:0` dentro del notebook.
4. Revisar las rutas del dataset desde Linux y repetir la auditoría de integridad antes de entrenar. No mover las imágenes ni distribuir capturas de un mismo fruto entre particiones.
5. Ejecutar una condición de entrenamiento a la vez. Comparar D1/D2 con el baseline disponible usando las mismas particiones y métricas.
6. Realizar fine-tuning parcial con control del consumo de memoria y conservar el conjunto de prueba fuera de la selección de variantes.

El cambio de dispositivo de CPU a GPU puede alterar ligeramente los resultados numéricos. Para comparar nuevos experimentos deben mantenerse documentados la semilla, el manifiesto de archivos, las transformaciones y los hiperparámetros.

## 9. Ubicación del registro en el proyecto

Guardar este notebook en:

```text
PROYECTO_TI_2/
`-- 1. fuentes/
    |-- bitacora_proyecto_clasificacion_palta.ipynb
    `-- preparacion_entorno_gpu_wsl2.ipynb
```

La documentación del entorno queda en `1. fuentes/`. El código de entrenamiento debe continuar en `6. entrenamiento/resnet50.ipynb` y sus resultados en `7. resultados/entrenamiento/`.

No es necesario volver a ejecutar este reporte para leerlo: las salidas de terminal ya están registradas. Las dos celdas de diagnóstico son opcionales y pueden utilizarse para futuras verificaciones.